# Customer Churn - Preprocessing

Dataset: [Real-World Churn (Kaggle)](https://www.kaggle.com/datasets/lasaljaywardena/real-world-churn/data)

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.metrics import roc_auc_score

pd.set_option('display.max_columns', 50)

In [2]:
RAW_DIR = 'data/raw'
OUT_DIR = 'data/processed'

N_DAYS = 90
DAY_COLS = ['Day_' + str(i) for i in range(1, 91)]

## 2. Load data
### 2.1 Labels

In [3]:
train_ids = pd.read_csv(RAW_DIR + '/train/train_cxid.csv')
test_ids = pd.read_csv(RAW_DIR + '/test/test_cxid.csv', encoding='utf-8-sig')  # test file has BOM
print(train_ids.shape, test_ids.shape)
train_ids.head()

(52004, 2) (13001, 2)


,customer_id,churn
0,a8cfed4379972f41e730a31a5c500e9ed2592374761b09...,0
1,d231e6c6dfd09290c16f640288edcd0ec5de526681c5e8...,0
2,787d994dc6e133f5711414ac7959d3263af560f19dd634...,0
3,db2a28328e3b7fc2684b7fc75838e45f5090c944ab8e8e...,0
4,2553072766402979a00e89a9413f4fc5a46e3d8f3e7d18...,0


In [4]:
print(train_ids.churn.mean(), test_ids.churn.mean())
print(len(set(train_ids.customer_id) & set(test_ids.customer_id)))

0.23117452503653566 0.2290593031305284
0


### 2.2 Usage data

In [5]:
usage = pd.read_csv(RAW_DIR + '/main.csv')
print(usage.shape)
usage.head()

(780060, 92)


,customer_id,usage_type,Day_1,Day_2,Day_3,Day_4,Day_5,Day_6,Day_7,Day_8,Day_9,Day_10,Day_11,Day_12,Day_13,Day_14,Day_15,Day_16,Day_17,Day_18,Day_19,Day_20,Day_21,Day_22,Day_23,...,Day_66,Day_67,Day_68,Day_69,Day_70,Day_71,Day_72,Day_73,Day_74,Day_75,Day_76,Day_77,Day_78,Day_79,Day_80,Day_81,Day_82,Day_83,Day_84,Day_85,Day_86,Day_87,Day_88,Day_89,Day_90
0,e9126f469646506b8e82b4214d075bf72d9d324d357650...,usage_app_other,53.00,130.73,7.86,67.71,153.38,35.92,128.60,30.61,37.29,86.45,0.00,77.17,0.00,64.13,122.66,27.82,0.00,10.13,18.90,0.00,0.00,0.00,41.38,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.00,0.0
1,e9126f469646506b8e82b4214d075bf72d9d324d357650...,usage_app_youtube_daily,98.50,0.00,284.82,150.05,0.00,261.89,44.89,0.00,0.00,0.00,0.00,0.00,23.67,0.00,0.00,0.00,0.00,96.61,38.97,0.00,91.72,0.00,55.94,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,50.28,0.0
2,e9126f469646506b8e82b4214d075bf72d9d324d357650...,usage_app_facebook_daily,79.42,72.61,72.15,82.82,61.06,57.91,55.59,56.46,50.07,47.65,34.87,43.82,30.76,23.70,20.36,21.03,25.00,27.88,18.49,24.19,11.80,0.00,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.00,0.0
3,e9126f469646506b8e82b4214d075bf72d9d324d357650...,usage_app_tiktok_daily,53.22,65.80,52.87,59.78,50.75,52.37,44.12,40.88,44.82,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,11.72,5.22,0.59,0.00,3.87,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.00,0.0
4,e9126f469646506b8e82b4214d075bf72d9d324d357650...,usage_app_whatsapp_daily,0.00,4.18,20.33,7.89,18.37,7.17,5.22,12.77,4.63,17.34,10.47,6.67,3.60,8.44,21.32,10.04,10.22,13.20,6.90,1.44,0.00,0.00,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.51,0.0,1.19,5.16,0.0,0.0,0.0,0.0,0.0,2.31,0.0,0.0,0.0,1.22,0.0


In [6]:
print(usage.customer_id.nunique())
print(usage[DAY_COLS].isna().sum().sum(), (usage[DAY_COLS] < 0).sum().sum())
usage.usage_type.value_counts()

65005


0 0


usage_type
usage_app_other              65005
usage_app_youtube_daily      65005
usage_app_facebook_daily     65005
usage_app_tiktok_daily       65005
usage_app_whatsapp_daily     65005
usage_app_helakuru_daily     65005
usage_voice_d2d_outgoing     65005
usage_voice_d2nd_outgoing    65005
usage_voice_d2d_incoming     65005
usage_voice_nd2d_incoming    65005
usage_pack_data              65005
usage_pack_vas               65005
Name: count, dtype: int64

In [7]:
usage.groupby('customer_id').size().value_counts()

12    65005
Name: count, dtype: int64

In [8]:
all_ids = pd.concat([train_ids, test_ids])
all_ids.customer_id.isin(usage.customer_id).mean()

np.float64(1.0)

## 3. Feature engineering
### 3.1 Features per usage type

In [ ]:
def series_features(df, n_days=N_DAYS):
    k = n_days // 3
    X = df[DAY_COLS[:n_days]].to_numpy(dtype=np.float32)
    act = X > 0
    f = pd.DataFrame()
    f['ratio_last30_first30'] = (X[:, -k:].mean(axis=1) + 1) / (X[:, :k].mean(axis=1) + 1)
    f['active_days'] = act.sum(axis=1)
    last = np.where(act.any(axis=1), n_days - 1 - np.argmax(act[:, ::-1], axis=1), -1)
    f['days_since_active'] = n_days - 1 - last
    f['customer_id'] = df.customer_id.values
    f['usage_type'] = df.usage_type.values
    return f

In [10]:
long_feats = series_features(usage)
long_feats.shape

(780060, 5)

### 3.2 One row per customer

In [11]:
def short_name(s):
    return s.replace('usage_', '').replace('_daily', '')

def to_wide(df):
    df = df.copy()
    df['usage_type'] = df['usage_type'].map(short_name)
    w = df.pivot(index='customer_id', columns='usage_type')
    w.columns = [u + '__' + f for f, u in w.columns]
    return w

In [12]:
wide_feats = to_wide(long_feats)
print(wide_feats.shape)
wide_feats.head()

(65005, 36)


,app_facebook__ratio_last30_first30,app_helakuru__ratio_last30_first30,app_other__ratio_last30_first30,app_tiktok__ratio_last30_first30,app_whatsapp__ratio_last30_first30,app_youtube__ratio_last30_first30,pack_data__ratio_last30_first30,pack_vas__ratio_last30_first30,voice_d2d_incoming__ratio_last30_first30,voice_d2d_outgoing__ratio_last30_first30,voice_d2nd_outgoing__ratio_last30_first30,voice_nd2d_incoming__ratio_last30_first30,app_facebook__active_days,app_helakuru__active_days,app_other__active_days,app_tiktok__active_days,app_whatsapp__active_days,app_youtube__active_days,pack_data__active_days,pack_vas__active_days,voice_d2d_incoming__active_days,voice_d2d_outgoing__active_days,voice_d2nd_outgoing__active_days,voice_nd2d_incoming__active_days,app_facebook__days_since_active,app_helakuru__days_since_active,app_other__days_since_active,app_tiktok__days_since_active,app_whatsapp__days_since_active,app_youtube__days_since_active,pack_data__days_since_active,pack_vas__days_since_active,voice_d2d_incoming__days_since_active,voice_d2d_outgoing__days_since_active,voice_d2nd_outgoing__days_since_active,voice_nd2d_incoming__days_since_active
customer_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0000f727854b50bb95c054b39c1fe5c92e5ebcfa4bcb5dc279f56aa96a365e5a,1.199179,1.467002,2.226951,2.190454,0.228659,1.600869,0.860780,3.941012,0.082361,0.988904,3.541094,0.715772,56,61,74,73,14,67,50,76,26,47,78,53,0,1,0,0,67,1,0,0,59,2,0,0
0001702d0abcdbeeb23f7c434aa6858030f365caf1464aa9c383837cba2f82a6,0.129684,0.576833,0.569635,4.126400,0.198754,0.257328,0.594390,0.113792,2.572188,0.080123,0.124818,0.552681,27,46,51,85,25,43,51,23,90,18,23,45,1,0,5,0,51,0,0,54,0,14,30,0
0001ae52a920c36edc7ab364ea58a58ca725f0ad44c02ba74ecefe08bb2b91a3,0.338972,0.523607,3.551736,0.961560,0.309249,0.545200,4.236990,0.407581,0.101685,0.753146,1.070638,0.400448,49,38,83,47,34,49,81,36,30,37,46,39,5,0,0,1,2,3,0,1,45,15,0,1
000205e54ddf8bc7512490895ce397647faaedc114f411b5c0a693b6f250617d,2.110224,1.028402,2.084546,3.021145,2.571347,3.193614,2.067336,2.674130,0.999107,2.044519,3.983816,3.533364,72,62,45,85,76,83,69,72,48,82,82,68,0,0,0,0,0,0,0,0,3,0,0,0
0002511664bff302e40b19ed6872ad4390ac59901e55e2c81a9cc02b21de783a,0.838070,1.433894,2.341501,2.270095,2.401353,2.147760,0.017154,2.138575,0.791873,3.778404,1.365859,0.661992,51,61,74,76,76,65,12,84,49,79,77,51,2,0,0,0,0,0,38,0,5,0,0,0


### 3.3 Cross-service features

In [13]:
def add_overall_features(w, usage_df, n_days=N_DAYS):
    k = n_days // 3
    w = w.copy()
    for feat in ['ratio_last30_first30', 'active_days', 'days_since_active']:
        cols = [c for c in w.columns if c.endswith('__' + feat)]
        w['all__' + feat + '_min'] = w[cols].min(axis=1)
        w['all__' + feat + '_mean'] = w[cols].mean(axis=1)
        w['all__' + feat + '_max'] = w[cols].max(axis=1)

    days = usage_df.set_index('customer_id')[DAY_COLS[:n_days]]
    svc = usage_df.usage_type.map(short_name)

    tot = days.groupby(level=0).sum().loc[w.index].values
    w['all__total'] = tot.sum(axis=1)
    w['all__last30_share'] = tot[:, -k:].sum(axis=1) / (tot.sum(axis=1) + 1)

    # data = apps + data pack
    row_tot = days.sum(axis=1)
    data = row_tot[svc.str.startswith(('app_', 'pack_data')).values].groupby(level=0).sum()
    voice = row_tot[svc.str.startswith('voice_').values].groupby(level=0).sum()
    w['data_voice_ratio'] = data / (voice + 1)

    w['n_services_active_last7'] = (days.iloc[:, -7:] > 0).any(axis=1).groupby(level=0).sum()
    return w

In [14]:
features = add_overall_features(wide_feats, usage)
print(features.shape)
features.filter(like='all__').describe().T

(65005, 49)


,count,mean,std,min,25%,50%,75%,max
all__ratio_last30_first30_min,65005.0,0.147130,0.223142,0.004461,0.023609,0.044997,0.147312,1.880725
all__ratio_last30_first30_mean,65005.0,1.472652,0.646768,0.086395,1.145917,1.647444,1.934799,3.202158
all__ratio_last30_first30_max,65005.0,3.327291,1.053929,0.182282,3.015192,3.582059,4.041404,9.661685
all__active_days_min,65005.0,21.980217,10.476696,5.000000,15.000000,18.000000,26.000000,67.000000
all__active_days_mean,65005.0,53.839866,12.347154,19.750000,48.250000,57.833333,62.666667,82.166667
all__active_days_max,65005.0,80.298177,11.455849,28.000000,80.000000,83.000000,87.000000,90.000000
all__days_since_active_min,65005.0,0.028121,0.355313,0.000000,0.000000,0.000000,0.000000,42.000000
all__days_since_active_mean,65005.0,10.575652,10.173200,0.000000,3.000000,6.916667,15.000000,57.833333
all__days_since_active_max,65005.0,48.457503,25.806188,0.000000,25.000000,59.000000,70.000000,81.000000
all__total,65005.0,61779.751267,23685.556769,7338.130000,46366.230000,65732.850000,78983.560000,143802.470000


## 4. Merge with labels and save

In [15]:
train_df = train_ids.merge(features, left_on='customer_id', right_index=True, how='left')
test_df = test_ids.merge(features, left_on='customer_id', right_index=True, how='left')
print(train_df.shape, train_df.isna().sum().sum())
print(test_df.shape, test_df.isna().sum().sum())

(52004, 51) 0
(13001, 51) 0


In [16]:
os.makedirs(OUT_DIR, exist_ok=True)
train_df.to_csv(OUT_DIR + '/train_features.csv', index=False)
test_df.to_csv(OUT_DIR + '/test_features.csv', index=False)

### 4.1 Single feature AUC

In [17]:
feature_cols = [c for c in train_df.columns if c not in ['customer_id', 'churn']]
auc = {}
for c in feature_cols:
    s = roc_auc_score(train_df.churn, train_df[c])
    auc[c] = max(s, 1 - s)
pd.Series(auc).sort_values(ascending=False).head(15)

all__ratio_last30_first30_mean               0.999465
all__active_days_mean                        0.999462
n_services_active_last7                      0.991796
all__days_since_active_mean                  0.987128
all__total                                   0.985678
all__last30_share                            0.972367
all__ratio_last30_first30_max                0.878118
all__ratio_last30_first30_min                0.867230
voice_d2d_incoming__ratio_last30_first30     0.856930
voice_nd2d_incoming__ratio_last30_first30    0.855654
app_helakuru__ratio_last30_first30           0.855253
pack_data__ratio_last30_first30              0.855181
voice_d2nd_outgoing__ratio_last30_first30    0.854248
voice_d2d_outgoing__ratio_last30_first30     0.853735
app_facebook__ratio_last30_first30           0.853614
dtype: float64

## 5. Features from the first 30 days only

The same 49 features, built from days 1-30 only, to see how early churn can be predicted. Column names are kept the same so the training code works unchanged, but within the 30-day window:
- ratio_last30_first30 = (mean usage days 21-30 + 1) / (mean usage days 1-10 + 1), i.e. last third vs first third
- active_days counts days 1-30 (0-30), days_since_active = 30 - last active day (30 = never used)
- all__last30_share = share of the 30-day total used in days 21-30
- n_services_active_last7 = services used in days 24-30

In [18]:
N_DAYS_SHORT = 30

wide_30 = to_wide(series_features(usage, n_days=N_DAYS_SHORT))
features_30 = add_overall_features(wide_30, usage, n_days=N_DAYS_SHORT)

train_30 = train_ids.merge(features_30, left_on='customer_id', right_index=True, how='left')
test_30 = test_ids.merge(features_30, left_on='customer_id', right_index=True, how='left')
print(train_30.shape, train_30.isna().sum().sum())
print(test_30.shape, test_30.isna().sum().sum())

train_30.to_csv(OUT_DIR + '/train_features_30d.csv', index=False)
test_30.to_csv(OUT_DIR + '/test_features_30d.csv', index=False)

(52004, 51) 0
(13001, 51) 0


In [19]:
auc_30 = {}
for c in feature_cols:
    s = roc_auc_score(train_30.churn, train_30[c])
    auc_30[c] = max(s, 1 - s)
pd.DataFrame({'AUC 90 days': pd.Series(auc), 'AUC 30 days': pd.Series(auc_30)}).sort_values('AUC 30 days', ascending=False).head(15).round(4)

,AUC 90 days,AUC 30 days
all__ratio_last30_first30_mean,0.9995,0.9244
all__days_since_active_mean,0.9871,0.8748
all__last30_share,0.9724,0.8338
n_services_active_last7,0.9918,0.8228
all__total,0.9857,0.8129
all__ratio_last30_first30_max,0.8781,0.7610
all__active_days_min,0.7858,0.7184
all__days_since_active_max,0.8336,0.6926
all__active_days_mean,0.9995,0.6914
pack_data__ratio_last30_first30,0.8552,0.6714
